# Microsoft Agent Framework — Lab 5: Agent Skills

**Agent Skills** are portable packages of instructions, scripts, and resources that give agents specialized domain expertise. They follow an open specification at [agentskills.io](https://agentskills.io/).

Skills solve a core problem: you can't put every piece of domain knowledge in the system prompt — it's too expensive and most of it is irrelevant to any given query. Skills use **progressive disclosure** to load context only when needed:

```
Stage 1 – Advertise  (~100 tokens per skill)
  → Skill names and descriptions are injected into the system prompt at startup

Stage 2 – Load
  → Agent calls load_skill("skill-name") to get full instructions on demand

Stage 3 – Read Resources
  → Agent calls read_skill_resource("skill-name", "resource-name") for supplementary files

Stage 4 – Run Scripts
  → Agent calls run_skill_script("skill-name", "script-name", args={...}) to execute code
```

This lab covers:
1. **File-based skills** — `SKILL.md` directories on disk
2. **Code-defined skills** — `Skill` objects in Python with `@skill.resource` and `@skill.script`
3. **Combining** file-based and code-defined skills
4. **Script approval** — human-in-the-loop before script execution
5. **Runtime injection** — forwarding context to skill resources and scripts

> **Experimental:** Skills are a new feature. Import them from `agent_framework` as normal — the `@experimental` marker just means the API may evolve.


In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

---
## Part 1 — File-Based Skills

A file-based skill is a directory containing a `SKILL.md` file. The framework discovers it automatically when you point `SkillsProvider` at the parent directory.

### Skill structure

```
skills/
└── travel-policy/
    ├── SKILL.md                          ← required: frontmatter + instructions
    ├── references/
    │   └── refund-policy.md              ← read via read_skill_resource
    └── scripts/
        └── estimate_refund.py            ← run via run_skill_script
```

### SKILL.md format

```yaml
---
name: travel-policy
description: Travel booking policies, refund rules, and passenger rights. Use when ...
---

## Instructions

Step-by-step guidance the agent follows when this skill is loaded.
```

The description is what the agent sees in the system prompt (~advertised). The body is what it loads on demand.


In [2]:
import os
import sqlite3
from pathlib import Path
from textwrap import dedent

from agent_framework import Skill, SkillResource, SkillScript, SkillsProvider
from agent_framework.openai import OpenAIChatCompletionClient
from IPython.display import display, Markdown

client = OpenAIChatCompletionClient(model="gpt-4o-mini")

# Point SkillsProvider at the skills/ directory next to this notebook
SKILLS_DIR = Path("skills")
print("Skills directory:", SKILLS_DIR.resolve())
print("Contents:", [p.name for p in SKILLS_DIR.iterdir() if p.is_dir()])


Skills directory: /home/niket/ai/course-ai-agents-donner-udemy/7_ms_agent_framework/skills
Contents: ['travel-policy']


In [3]:
# Peek at what we built in skills/travel-policy/
print("=== SKILL.md ===")
print((SKILLS_DIR / "travel-policy" / "SKILL.md").read_text())
print("\n=== references/refund-policy.md (first 10 lines) ===")
lines = (SKILLS_DIR / "travel-policy" / "references" / "refund-policy.md").read_text().splitlines()
print("\n".join(lines[:10]))


=== SKILL.md ===
---
name: travel-policy
description: Travel booking policies, refund rules, and passenger rights. Use when customers ask about booking changes, cancellations, refunds, rebooking fees, or their rights during delays and disruptions.
---

## Booking Policy

When assisting customers with travel policies, follow these steps:

1. Identify whether the query is about booking changes, cancellations, refunds, or disruptions.
2. Read the `refund-policy` resource for detailed rules before answering.
3. Provide clear, accurate information and quote the relevant rule where helpful.
4. Always confirm whether the customer needs further assistance.

## Key Rules (summary — always read the refund-policy resource for full details)

- Bookings are changeable up to 48 hours before departure for a $50 change fee.
- Cancellations within 24 hours of booking receive a full refund, no questions asked.
- Cancellations 7+ days before travel receive a 75% refund.
- Cancellations within 7 days of t

In [5]:
import subprocess
import sys

def subprocess_runner(skill, script, args: dict | None = None) -> str:
    """Run a .py script as a local subprocess. Demo only — add sandboxing in production."""
    script_path = Path(skill.path) / script.path
    cmd = [sys.executable, str(script_path)]
    if args:
        cmd.extend(str(v) for v in args.values())
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
    return result.stdout.strip()


In [6]:
# Create a SkillsProvider and attach it to an agent via context_providers=
skills_provider = SkillsProvider(skill_paths=SKILLS_DIR, script_runner=subprocess_runner)

policy_agent = client.as_agent(
    name="PolicyAgent",
    instructions=(
        "You are a helpful airline support agent. "
        "Use your skills to answer customer questions accurately."
    ),
    context_providers=[skills_provider],
)

# Ask a policy question — watch the agent call load_skill and read_skill_resource
result = await policy_agent.run(
    "I booked a flight 3 days ago and need to cancel. "
    "My flight is in 5 days. How much refund will I get?"
)
display(Markdown(str(result)))


Based on our travel policy, since you booked your flight 3 days ago and are looking to cancel it within 5 days of travel, the cancellation terms apply as follows:

- Cancellations made within 7 days of the flight are typically non-refundable, which means you would not receive a refund.

There are exceptions to this policy for medical reasons, but without specific circumstances qualifying for those exceptions, a refund will not be available.

If you have any further questions or need additional assistance, please let me know!

### What just happened?

The framework injected the `travel-policy` skill's **name and description** into the system prompt. When the agent recognised the query matched the skill's domain, it:

1. Called `load_skill("travel-policy")` → received the full SKILL.md instructions
2. Called `read_skill_resource("travel-policy", "refund-policy")` → fetched the policy table
3. Used that context to answer accurately

No extra tokens were spent until the skill was actually needed.


---
## Part 2 — File-Based Scripts

Scripts bundled inside a skill directory let the agent **run code** as part of its answer. The agent calls `run_skill_script` automatically when the skill instructs it to.

To enable file-based scripts, pass a `script_runner` callable to `SkillsProvider`. The runner receives the resolved skill and script objects and executes the file.

> **Security note:** The demo runner below uses `subprocess` for simplicity. In production, add sandboxing, timeouts, and input validation.


In [7]:
import subprocess
import sys
from agent_framework import Skill, SkillScript

def subprocess_runner(skill: Skill, script: SkillScript, args: dict | None = None) -> str:
    """Run a .py script as a local subprocess. Demo only — add sandboxing in production."""
    script_path = Path(skill.path) / script.path  # type: ignore[arg-type]
    cmd = [sys.executable, str(script_path)]
    if args:
        # Pass each value as a positional argument (script reads sys.argv)
        cmd.extend(str(v) for v in args.values())
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
    return result.stdout.strip()

# Re-create provider with the script runner
skills_provider_with_scripts = SkillsProvider(
    skill_paths=SKILLS_DIR,
    script_runner=subprocess_runner,
)

scripted_agent = client.as_agent(
    name="RefundAgent",
    instructions=(
        "You are an airline refund specialist. "
        "Use your skills to calculate and explain refunds accurately."
    ),
    context_providers=[skills_provider_with_scripts],
)

result = await scripted_agent.run(
    "My ticket cost $499 and my flight is in 10 days. "
    "How much will I get back if I cancel now?"
)
display(Markdown(str(result)))


Based on the general travel policies, if you cancel your ticket costing $499 and your flight is in 10 days, you will receive a 75% refund since you are canceling more than 7 days before your flight. 

Here's the calculation:
- Ticket Cost: $499
- Refund Percentage: 75%
- Refund Amount: $499 * 0.75 = $374.25

So, you would get back approximately **$374.25** after canceling your ticket. 

If you have any other questions or need further assistance, feel free to ask!

---
## Part 3 — Code-Defined Skills

Code-defined skills live entirely in Python — no files needed. They're useful when skill content is dynamic (e.g., read from a database) or when you want to keep skills alongside application code.

The three building blocks:

| Class / Decorator | Purpose |
|---|---|
| `Skill(name, description, content)` | Core skill object with instructions |
| `SkillResource(name, content=...)` | Static reference document |
| `@skill.resource` | Dynamic resource — called fresh each time the agent reads it |
| `@skill.script(name, description)` | In-process executable script |


In [8]:
DB_PATH = "tickets.db"

def get_city_price_raw(city: str) -> float | None:
    if not os.path.exists(DB_PATH):
        return None
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None


# ── Build the code-defined skill ─────────────────────────────────────────────

pricing_skill = Skill(
    name="pricing-advisor",
    description=(
        "Flight pricing, cost estimates, and value comparisons. "
        "Use when customers ask about prices, budget planning, or trip cost calculations."
    ),
    content=dedent("""\
        ## Pricing Advisor

        Use this skill to answer any question about flight pricing or trip budgeting.

        1. Read the `currency-table` resource for exchange rates.
        2. Read the `destinations` resource for live prices from the database.
        3. Use the `calculate-trip-cost` script to compute total trip cost including hotel.
        4. Always show the currency conversion if the customer asks for non-USD pricing.
    """),
    resources=[
        # Static resource — always the same content
        SkillResource(
            name="currency-table",
            description="Exchange rates from USD to common travel currencies",
            content=dedent("""\
                | Currency | Symbol | Rate (per USD) |
                |---|---|---|
                | Euro | EUR | 0.92 |
                | British Pound | GBP | 0.79 |
                | Japanese Yen | JPY | 149.50 |
                | Australian Dollar | AUD | 1.53 |
                | Canadian Dollar | CAD | 1.36 |
            """),
        ),
    ],
)


# Dynamic resource — reads from the database each time
@pricing_skill.resource(name="destinations", description="Live prices for all destinations from the database")
def get_destinations() -> str:
    """Return current prices from the live database."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT city_name, round_trip_price FROM cities ORDER BY round_trip_price")
    rows = c.fetchall()
    conn.close()
    if not rows:
        return "No destinations available."
    lines = ["| Destination | Round-Trip (USD) |", "|---|---|"]
    lines += [f"| {row[0].title()} | ${row[1]:.0f} |" for row in rows]
    return "\n".join(lines)

print("Skill resources:", [r.name for r in pricing_skill.resources])


Skill resources: ['currency-table', 'destinations']


In [9]:
import json
from typing import Any

# In-process script — called directly, no subprocess needed
@pricing_skill.script(name="calculate-trip-cost", description="Calculate total trip cost: flights + hotel nights")
def calculate_trip_cost(city: str, hotel_per_night: float, nights: int, **kwargs: Any) -> str:
    """Calculate total trip cost including flights and hotel."""
    flight = get_city_price_raw(city)
    if flight is None:
        return json.dumps({"error": f"No flight price found for {city}"})
    hotel_total = hotel_per_night * nights
    total = flight + hotel_total
    currency = kwargs.get("currency", "USD")
    rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 149.50, "AUD": 1.53, "CAD": 1.36}
    rate = rates.get(currency.upper(), 1.0)
    return json.dumps({
        "city": city.title(),
        "flights_usd": flight,
        "hotel_per_night_usd": hotel_per_night,
        "nights": nights,
        "hotel_total_usd": hotel_total,
        "total_usd": total,
        "total_converted": round(total * rate, 2),
        "currency": currency,
    })

print("Skill scripts:", [s.name for s in pricing_skill.scripts])

# Run it manually to verify before attaching to an agent
raw = calculate_trip_cost("paris", hotel_per_night=180, nights=4)
print("Script test:", json.loads(raw))


Skill scripts: ['calculate-trip-cost']
Script test: {'city': 'Paris', 'flights_usd': 399.0, 'hotel_per_night_usd': 180, 'nights': 4, 'hotel_total_usd': 720, 'total_usd': 1119.0, 'total_converted': 1119.0, 'currency': 'USD'}


In [10]:
# Attach the code-defined skill to an agent
code_skills_provider = SkillsProvider(skills=[pricing_skill])

pricing_agent = client.as_agent(
    name="PricingAdvisor",
    instructions=(
        "You are a travel pricing expert. "
        "Help customers understand costs and plan trips within their budget."
    ),
    context_providers=[code_skills_provider],
)

result = await pricing_agent.run(
    "I want to visit Paris for 5 nights. Hotels run about $180/night. "
    "What's my total trip cost in Euros?"
)
display(Markdown(str(result)))


Your total trip cost for 5 nights in Paris, including flights and hotel accommodations, is approximately **$1,299 USD**.

To convert this amount to Euros at the current exchange rate of **0.92 EUR per USD**:
- **Total in Euros: €1,192.08 EUR**

So, your estimated total trip cost is about **€1,192.08**.

---
## Part 4 — Combining File-Based and Code-Defined Skills

Pass both `skill_paths` and `skills` to a single `SkillsProvider`. The agent sees all skills in its system prompt and loads whichever is relevant to the query.


In [11]:
combined_provider = SkillsProvider(
    skill_paths=SKILLS_DIR,          # discovers: travel-policy
    skills=[pricing_skill],          # adds:       pricing-advisor
    script_runner=subprocess_runner, # needed for file-based scripts
)

# Show what skills the provider knows about
print("Skills registered:")
for name, skill in combined_provider._skills.items():
    source = "file-based" if skill.path else "code-defined"
    print(f"  {name!r} ({source}): {skill.description[:60]}...")

combined_agent = client.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a full-service travel agent. "
        "Use your skills to answer questions about both pricing and policies."
    ),
    context_providers=[combined_provider],
)

# One query touches pricing, another touches policy
print("\n=== Pricing query ===")
r1 = await combined_agent.run("How much would a 3-night trip to Rome cost if hotels are $220/night?")
display(Markdown(str(r1)))

print("\n=== Policy query ===")
r2 = await combined_agent.run("If I cancel a $499 ticket 9 days before my flight, what refund do I get?")
display(Markdown(str(r2)))


Skills registered:
  'travel-policy' (file-based): Travel booking policies, refund rules, and passenger rights....
  'pricing-advisor' (code-defined): Flight pricing, cost estimates, and value comparisons. Use w...

=== Pricing query ===


A 3-night trip to Rome would cost approximately $1,159 in total. This includes:

- Hotel cost: $660 ($220 per night for 3 nights)
- Flights: $499 

The total trip cost is $1,159.


=== Policy query ===


If you cancel a $499 ticket 9 days before your flight, you will receive a 75% refund of the ticket price. This means you would get back approximately $374.25.

If you have any further questions or need assistance with the cancellation process, feel free to ask!

---
## Part 5 — Script Approval (Human-in-the-Loop)

Set `require_script_approval=True` on `SkillsProvider` to gate every script execution behind human confirmation. The agent pauses and returns a `user_input_requests` list — your code presents the call to the human and passes back an approval or denial.

This is critical for scripts with real-world side effects (email, payments, deployments).


In [24]:
from agent_framework import InMemoryHistoryProvider

approval_provider = SkillsProvider(
    skill_paths=SKILLS_DIR,
    skills=[pricing_skill],
    script_runner=subprocess_runner,   # for the file-based estimate_refund.py
    require_script_approval=True,      # all scripts now require human approval
)

approval_agent = client.as_agent(
    name="ApprovalAgent",
    instructions=(
        "You are a travel agent. Use your skills to help customers. "
        "Always calculate refunds precisely using the estimate_refund script."
    ),
    # Approval responses become tool messages. The history provider keeps
    # the preceding assistant tool_calls message in the same session.
    context_providers=[approval_provider, InMemoryHistoryProvider()],
)

approval_prompt = "My $499 ticket is 10 days away. I want to cancel — run the refund calculator."
approval_consumed = False
session = approval_agent.create_session()

print("=== Step 1: Initial run ===")
result = await approval_agent.run(
    approval_prompt,
    session=session,
)

print(f"user_input_requests: {len(result.user_input_requests)}")
for req in result.user_input_requests:
    fc = getattr(req, "function_call", None)
    if fc:
        print(f"  Script: {getattr(fc, 'name', '?')}")
        print(f"  Args:   {getattr(fc, 'arguments', '?')}")


=== Step 1: Initial run ===
user_input_requests: 2
  Script: read_skill_resource
  Args:   {"skill_name": "travel-policy", "resource_name": "refund-policy"}
  Script: run_skill_script
  Args:   {"skill_name": "travel-policy", "script_name": "estimate_refund", "args": {"ticket_price": 499, "days_until_departure": 10}}


In [25]:
# Step 2: Human approves the script execution
def has_pending_approval_history(session) -> bool:
    messages = session.state.get("in_memory", {}).get("messages", [])
    return any(
        any(content.type == "function_approval_request" for content in msg.contents)
        for msg in messages
    )


if result.user_input_requests:
    if approval_consumed:
        raise RuntimeError(
            "This approval response has already been submitted. "
            "Re-run the Step 1 cell to create a fresh approval request."
        )
    if not has_pending_approval_history(session):
        raise RuntimeError(
            "The session is missing the assistant tool-call history for this approval. "
            "Re-run the Step 1 cell immediately before approving."
        )

    print("Human decision: APPROVE\n")
    approvals = [req.to_function_approval_response(approved=True) for req in result.user_input_requests]

    final = await approval_agent.run(
        approvals,
        session=session,
    )
    approval_consumed = True

    print("=== Final result ===")
    #display(Markdown(str(final)))
    display(Markdown(final.text))
    print(final.text)


Human decision: APPROVE

=== Final result ===


---
## Part 6 — Runtime Injection via `function_invocation_kwargs`

Skill resource and script functions that accept `**kwargs` automatically receive values passed to `agent.run(..., function_invocation_kwargs={...})`. This lets skills access per-request context (user preferences, currency choice, auth tokens) without hardcoding them.


In [18]:
# Add **kwargs to the resource and script so they receive runtime context
runtime_skill = Skill(
    name="personalised-pricing",
    description=(
        "Personalised trip cost calculator that uses the customer's preferred currency. "
        "Use when a customer asks for trip costs."
    ),
    content=dedent("""\
        ## Personalised Pricing

        1. Read the `destinations` resource to get live prices.
        2. Use the `calculate-cost` script to compute total trip cost.
        3. Always present prices in the customer's preferred currency.
    """),
)

@runtime_skill.resource(name="destinations", description="Live destination prices in the customer's currency")
def runtime_destinations(**kwargs: Any) -> str:
    """Return prices converted to the customer's preferred currency."""
    currency = kwargs.get("currency", "USD")
    rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 149.50, "AUD": 1.53, "CAD": 1.36, "USD": 1.0}
    rate = rates.get(currency.upper(), 1.0)

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT city_name, round_trip_price FROM cities ORDER BY round_trip_price")
    rows = c.fetchall()
    conn.close()

    lines = [f"| Destination | Round-Trip ({currency}) |", "|---|---|"]
    lines += [f"| {row[0].title()} | {currency} {row[1] * rate:.0f} |" for row in rows]
    return "\n".join(lines)

@runtime_skill.script(name="calculate-cost", description="Calculate trip cost in the customer's preferred currency")
def runtime_calculate_cost(city: str, hotel_per_night: float, nights: int, **kwargs: Any) -> str:
    """Calculate trip cost, converting to the customer's currency."""
    currency = kwargs.get("currency", "USD")
    rates = {"EUR": 0.92, "GBP": 0.79, "JPY": 149.50, "AUD": 1.53, "CAD": 1.36, "USD": 1.0}
    rate = rates.get(currency.upper(), 1.0)

    flight = get_city_price_raw(city)
    if flight is None:
        return json.dumps({"error": f"No price for {city}"})
    total_usd = flight + (hotel_per_night * nights)
    return json.dumps({
        "city": city.title(),
        "total_usd": total_usd,
        "currency": currency,
        "total": round(total_usd * rate, 2),
    })

# Create a provider and agent using the runtime-aware skill
runtime_provider = SkillsProvider(skills=[runtime_skill])
runtime_agent = client.as_agent(
    name="PersonalisedAgent",
    instructions="You are a travel pricing agent. Use your skills to help customers.",
    context_providers=[runtime_provider],
)

# Pass the customer's currency preference at runtime — the skill reads it via kwargs
result = await runtime_agent.run(
    "What's the total cost for 4 nights in London at $180/night? Show me in GBP.",
    function_invocation_kwargs={"currency": "GBP"},
)
display(Markdown(str(result)))


The total cost for 4 nights in London at $180 per night is $720. To convert that to GBP, the approximate cost is £590.68.

---
### Summary

| Concept | Code |
|---|---|
| File-based skill | `SKILL.md` in a directory; `SkillsProvider(skill_paths=...)` |
| Code-defined skill | `Skill(name, description, content)` |
| Static resource | `SkillResource(name=..., content="...")` |
| Dynamic resource | `@skill.resource def f() -> Any: ...` |
| In-process script | `@skill.script def f(param: type) -> str: ...` |
| File-based script | `.py` in `scripts/` + `script_runner=my_runner` |
| Attach to agent | `client.as_agent(..., context_providers=[skills_provider])` |
| Script approval | `SkillsProvider(..., require_script_approval=True)` |
| Runtime context | `**kwargs` on resource/script + `agent.run(..., function_invocation_kwargs={...})` |

### Skills vs other patterns in this course

| | Agent Skills | Workflows (Labs 3–4) | Middleware (Lab 6) |
|---|---|---|---|
| **Controls** | What the agent *knows* | What the agent *does* | How agent calls *behave* |
| **Best for** | Domain knowledge, policies, reusable expertise | Deterministic pipelines, side effects | Logging, security, approval |
| **Discovery** | LLM decides when to load | Explicit execution graph | Transparent to agent |
